Input: a text in Ukrainian, Russian or mixed code, usually a Telegram post. Output: a list of location strings exactly as written in the text. Locations are cities, countries, regions, districts, streets and addresses, and other geographic objects. For example, `Сьогодні з Києва до Хелма вперше вирушив новий поїзд` should return `["Києва", "Хелма"]`. Scoring is F1 over sets of strings.


## What we work with

| File | What it is | Size | Labels |
|---|---|---|---|
| `uk_geo_dataset.csv` | news sentences, Ukr-Synth corpus | 1.01M | machine LOC/ORG/PER spans |
| `ru_geo_dataset.csv` | Lenta.ru news sentences, Nerus corpus | 8.03M | machine LOC/ORG/PER spans |
| `uk_geo_dataset_processed_v1.parquet` | tokens of the same uk corpus | 1.01M | flat O/LOC, no B/I |
| `labeling_sample.csv` | Telegram posts | 26 | manual |
| `test.csv` | Telegram posts | 477 | none |

Train is large, but it is news with machine labels. We are scored on Telegram, and only 26 posts have manual labels.

In [1]:
import ast, json, re, string, unicodedata
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

DATA = Path("../data")
pd.set_option("display.max_colwidth", 120, "display.max_rows", 100)

In [2]:
# Official metric of cometition
def hard_processing(text):
    text = re.sub(r"[{re.escape(string.punctuation)}]", "", text)
    text = re.sub(r"\d", "", text)
    text = re.sub(r"\b\w\b\s?", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text.lower()


def comp_metric(y_true, y_pred):
    assert len(y_true) == len(y_pred)
    tp, fp, fn, p = 0.0, 0.0, 0.0, 0.0
    for y_true_sample, y_pred_sample in zip(y_true, y_pred):
        y_true_sample = {hard_processing(s) for s in y_true_sample}
        y_pred_sample = {hard_processing(s) for s in y_pred_sample}
        tp += len(y_true_sample & y_pred_sample)
        fp += len(y_pred_sample - y_true_sample)
        fn += len(y_true_sample - y_pred_sample)
        p += len(y_true_sample)
    precision = tp / (tp + fp) if tp + fp else float(p == 0)
    recall = tp / (tp + fn) if tp + fn else float(p == 0)
    return round(2 * precision * recall / (precision + recall), 8) if precision + recall else 0

In [3]:
TOKEN = re.compile(r"\w+(?:['’ʼ]\w+)*|[^\w\s]")
SPAN = re.compile(r"\((\d+),\s*(\d+)\)")


def parse_spans(value):
    return [(int(a), int(b)) for a, b in SPAN.findall(str(value))]


def tokenize(text):
    return [m.span() for m in TOKEN.finditer(text)]


def clean_spans(text, spans):
    # Merges spans that touch or overlap (a name marked in pieces: Івано + - + Франківськ),
    # trims edge whitespace and drops spans without letters. It never splits a span.
    merged = []
    for a, b in sorted(spans):
        if merged and a <= merged[-1][1]:
            merged[-1] = (merged[-1][0], max(b, merged[-1][1]))
        else:
            merged.append((a, b))
    cleaned = []
    for a, b in merged:
        chunk = text[a:b]
        a, b = a + len(chunk) - len(chunk.lstrip()), b - (len(chunk) - len(chunk.rstrip()))
        if a < b and any(c.isalnum() for c in text[a:b]):
            cleaned.append((a, b))
    return cleaned


def spans_to_bio(offsets, spans):
    tags, misaligned = ["O"] * len(offsets), 0
    for a, b in spans:
        hit = [i for i, (s, e) in enumerate(offsets) if s < b and a < e]
        if not hit or offsets[hit[0]][0] != a or offsets[hit[-1]][1] != b:
            misaligned += 1
        if hit:
            tags[hit[0]] = "B"
            for i in hit[1:]:
                tags[i] = "I"
    return tags, misaligned


def bio_to_strings(text, offsets, tags, merge_adjacent=False):
    out, i = [], 0
    while i < len(tags):
        if tags[i] == "O":
            i += 1
            continue
        j = i + 1
        while j < len(tags) and (tags[j] == "I" or (merge_adjacent and tags[j] != "O")):
            j += 1
        out.append(text[offsets[i][0]:offsets[j - 1][1]])
        i = j
    return out

## 1. Дані та їхня якість


In [4]:
from IPython.display import display

uk = pd.read_csv(DATA / "uk_geo_dataset.csv")
test = pd.read_csv(DATA / "test.csv")
sample = pd.read_csv(DATA / "labeling_sample.csv")
pd.DataFrame({
    name: {"рядків": len(frame), "колонки": ", ".join(frame.columns)}
    for name, frame in {"uk": uk, "test": test, "labeling_sample": sample}.items()
}).T

,рядків,колонки
uk,1010000,"text, loc_markers, org_markers, per_markers, is_valid"
test,477,"text_id, text, locations"
labeling_sample,26,"text, locations, loc_markers"


| Колонка | Опис |
|---|---|
| `text` | Текст, у якому потрібно знайти локації. |
| `loc_markers` | Список координат локацій у тексті: `(початок, кінець)`.|
| `org_markers` | Координати згадок організацій.|
| `per_markers` | Координати згадок людей. |
| `locations` | Список локацій словами, наприклад `["Проспект Бандери"]`. У `labeling_sample.csv` це надані відповіді. У `test.csv` всі значення `[]` — порожні заготовки для прогнозів, а не позначка відсутності локацій. |
| `is_valid` | Позначка поділу українського датасету: `0` — навчання, `1` — валідація. |
| `text_id` | Ідентифікатор тексту в test, за яким можна зіставити текст із прогнозом моделі. |

**Що таке ru dataset?**

`ru_geo_dataset.csv` — російськомовні новинні речення з розміткою сутностей. За описом у `data/README.md`, набір сформований на основі Nerus. Один рядок — одне речення.

| Колонка | Що містить |
|---|---|
| `text` | Саме речення. |
| `loc_markers` | Координати згадок локацій у реченні. |
| `doc_id` | Ідентифікатор документа, з якого взято речення. |
| `sent_id` | Номер речення всередині документа. |

У CSV також є `org_markers` і `per_markers` — координати організацій та людей. Для поточного аналізу локацій завантажуємо чотири колонки з таблиці.


In [5]:
ru = pd.read_csv(DATA / "ru_geo_dataset.csv", usecols=["text", "loc_markers", "doc_id", "sent_id"])
ru_docs = ru.doc_id.to_numpy()
pd.Series({"рядків ru": len(ru), "документів": ru.doc_id.nunique()}).to_frame("кількість")


,кількість
рядків ru,8028840
документів,739190


**Чи є незаповнені комірки?**

Шукаємо рядки, де взагалі не записали текст або значення `loc_markers`. Наприклад, текст є, а комірку з маркерами залишили порожньою. Такі пропуски pandas позначає як `NaN`.

Запис `[]` — це заповнена комірка: у ній записано порожній список. Тому ця перевірка її не вважає пропуском. Чи є в самому тексті місця, які забули розмітити, цей код не перевіряє.


In [6]:
display(pd.DataFrame({
    name: frame[["text", "loc_markers"]].isna().sum()
    for name, frame in {"uk": uk, "ru": ru, "labeling_sample": sample}.items()
}).T)
print("Пропуски text у test:", int(test.text.isna().sum()))

,text,loc_markers
uk,0,0
ru,0,0
labeling_sample,0,0


Пропуски text у test: 0


**Висновок.** Незаповнених комірок у перевірених полях немає. Це означає лише, що значення записані; повноту розмітки локацій ми цим не перевірили. Нижче — реальні приклади, які пояснюють різницю.


**Місце в тексті є, але `loc_markers` дорівнює `[]`: як це виглядає?**

Показуємо два реальні рядки з навчальних CSV — один uk та один ru. Колонка «згадки без LOC-мітки» пояснює, на які слова дивитися; вихідну розмітку не змінюємо. `row_id` — індекс рядка після читання CSV, починаючи з нуля.


In [ ]:
unmarked_examples = pd.concat([
    uk.loc[[420657], ["text", "loc_markers"]].assign(dataset="uk", row_id=420657),
    ru.loc[[1748213], ["text", "loc_markers"]].assign(dataset="ru", row_id=1748213),
], ignore_index=True)
unmarked_examples["згадки без LOC-мітки"] = [["києві"], ["Москвы", "Санкт-Петербурга"]]

# These examples precede parsing: loc_markers still contains the original CSV strings.
assert unmarked_examples.loc_markers.eq("[]").all()
assert not unmarked_examples[["text", "loc_markers"]].isna().any().any()
assert all(place in text for text, places in zip(unmarked_examples.text, unmarked_examples["згадки без LOC-мітки"]) for place in places)
with pd.option_context("display.max_colwidth", None):
    display(unmarked_examples[["dataset", "row_id", "text", "loc_markers", "згадки без LOC-мітки"]])


,dataset,row_id,text,loc_markers,згадки без LOC-мітки
0,uk,420657,Півзахисник Шахтаря Тарас Степаненко прокоментував ситуацію з пошуком житла у києві працівниками клубу.,[],[києві]
1,ru,1748213,"Фанаты из Москвы, Санкт-Петербурга и других городов будут на одной стороне.",[],"[Москвы, Санкт-Петербурга]"


**Висновок.** У першому реченні йдеться про житло у Києві, у другому — про вболівальників із Москви та Санкт-Петербурга. Проте в обох рядках список LOC-маркерів порожній. Перевірка незаповнених комірок цього не виявляє: значення `[]` записане, але розмітка пропускає згадки місць.

Це два приклади неповної розмітки, а не підрахунок усіх таких випадків. Не кожен `[]` є помилкою: багато текстів справді не містять локацій.


**Як читати координати локацій?**

Перетворюємо рядкові маркери CSV на списки пар координат. Розмітку не виправляємо; показуємо один приклад із наданого sample.

In [7]:
for frame, column in [(uk, "loc_markers"), (uk, "org_markers"), (ru, "loc_markers"), (sample, "loc_markers")]:
    frame[column] = frame[column].map(parse_spans)
sample["locations"] = sample["locations"].map(ast.literal_eval)
row = sample[sample.loc_markers.map(bool)].iloc[0]
pd.DataFrame({"координати": row.loc_markers, "текст спану": [row.text[a:b] for a, b in row.loc_markers]})

,координати,текст спану
0,"(0, 16)",Проспект Бандери


**Висновок.** Пара `(a, b)` задає фрагмент `text[a:b]`: початок включно, кінець не включно. Кожна пара — окрема згадка; ці межі потрібні для побудови BIO.

**Чи виходять розібрані спани за межі тексту?**

Для кожного LOC-спану перевіряємо `0 ≤ початок < кінець ≤ довжина тексту`. Це перевірка координат, а не правильності вибору локації.

In [8]:
span_bounds = pd.DataFrame({
    name: {
        "спанів": int(frame.loc_markers.map(len).sum()),
        "недопустимі координати": sum(not (0 <= a < b <= len(text)) for text, spans in zip(frame.text, frame.loc_markers) for a, b in spans),
    }
    for name, frame in {"uk": uk, "ru": ru, "labeling_sample": sample}.items()
}).T
display(span_bounds)
assert span_bounds["недопустимі координати"].eq(0).all()

,спанів,недопустимі координати
uk,345114,0
ru,3708173,0
labeling_sample,32,0


**Висновок.** У розібраних LOC-маркерах uk, повного ru та sample недопустимих координат не виявлено. Це ще не означає, що кожен виділений фрагмент справді є локацією.

**Які підозрілі межі є в розмітці?**

`no_letters_or_digits` — спан без букв і цифр; `edge_space_or_punct` — пробіл або пунктуація на краю; `mid_word_boundary` — межа всередині слова. Рахуємо ці ознаки окремо, бо вони можуть перетинатися.

In [9]:
def marker_defects(df):
    c = Counter()
    for text, spans in zip(df["text"], df["loc_markers"]):
        for a, b in spans:
            s = text[a:b]
            c["spans"] += 1
            c["no_letters_or_digits"] += not any(ch.isalnum() for ch in s)
            c["edge_space_or_punct"] += not (s[0].isalnum() and s[-1].isalnum())
            c["mid_word_boundary"] += (a > 0 and text[a - 1].isalnum() and s[0].isalnum()) or (b < len(text) and text[b].isalnum() and s[-1].isalnum())
    return c


defects = pd.DataFrame({"uk": marker_defects(uk), "ru": marker_defects(ru), "sample": marker_defects(sample)}).fillna(0).astype(int)
defects

,uk,ru,sample
spans,345114,3708173,32
no_letters_or_digits,1057,50,0
edge_space_or_punct,5364,16792,0
mid_word_boundary,1516,7712,1


**Висновок.** В uk є 1 057 спанів без букв і цифр, 5 364 із пробілами або пунктуацією на краях та 1 516 із межею всередині слова. У повному ru відповідні числа — 50, 16 792 та 7 712. Це кандидати на перевірку, а не три незалежні списки доведених помилок.

**Що змінює поточна `clean_spans`?**

Застосовуємо наявну функцію до окремої колонки `spans`: вона об’єднує спани, що торкаються або перекриваються, обрізає крайові пробіли й відкидає фрагменти без букв і цифр. Оригінальні `loc_markers` зберігаємо.

In [10]:
uk["spans"] = [clean_spans(t, s) for t, s in zip(uk.text, uk.loc_markers)]
ru["spans"] = [clean_spans(t, s) for t, s in zip(ru.text, ru.loc_markers)]
sample["spans"] = sample["loc_markers"]
pd.DataFrame({
    name: {
        "спанів до": int(frame.loc_markers.map(len).sum()),
        "спанів після": int(frame.spans.map(len).sum()),
        "зменшення кількості": int(frame.loc_markers.map(len).sum() - frame.spans.map(len).sum()),
        "рядків зі змінами": sum(a != b for a, b in zip(frame.loc_markers, frame.spans)),
    }
    for name, frame in {"uk": uk, "ru": ru}.items()
}).T

,спанів до,спанів після,зменшення кількості,рядків зі змінами
uk,345114,341453,3661,2967
ru,3708173,3706582,1591,1569


**Висновок.** В uk кількість спанів зменшується на 3 661. Це різниця кількостей до/після, а не число виправлених помилок: злиття трьох частин в одну зменшує кількість на два, а обрізання пробілу не змінює її взагалі. У повному ru кількість спанів зменшується на 1 591, а розмітка змінюється у 1 569 рядках.

**Як виглядають зміни розмітки на реальних текстах?**

Беремо перші шість українських рядків, де `loc_markers` і `spans` відрізняються, та порівнюємо вирізані фрагменти.

In [11]:
changed = uk.loc[[a != b for a, b in zip(uk.loc_markers, uk.spans)]].head(6)
with pd.option_context("display.max_colwidth", None):
    display(pd.DataFrame({
        "text": changed.text,
        "до": [[t[a:b] for a, b in spans] for t, spans in zip(changed.text, changed.loc_markers)],
        "після": [[t[a:b] for a, b in spans] for t, spans in zip(changed.text, changed.spans)],
    }))

,text,до,після
335,З 2010 року КП «Полігон ТПВ» також займається відловом і стерилізацією безпритульних тварин в м. Івано-Франківську.,"[Івано, -, Франківську]",[Івано-Франківську]
1322,Субота та неділя в Івано-Франківську будуть хмарними.,"[Івано, -Франківську]",[Івано-Франківську]
1728,"Російські спецпризначенці беруть участь в атаках на аеропорт в Донецьку, - штаб АТО Бойовики продовжують атакувати аеропорт ""Донецьк"".","[Донецьку, , Донецьк""]","[Донецьку,, Донецьк""]"
1813,"Журналісти, які діставатимуться до аеропорту своїми транспортними засобами, повинні до 14-45 приїхати до адміністративної будівлі Аеропорту (біля готелю «Бориспіль»).","[«, Бориспіль»]",[«Бориспіль»]
2158,В Івано-Франківську ці гроші передають на потреби благодійної їдальні.,"[Івано, -Франківську]",[Івано-Франківську]
2410,Тіло артиста в орендованому ним домі в Санкт-Петербурзі знайшла його дружина.,"[Санкт, -Петербурзі]",[Санкт-Петербурзі]


**Висновок.** Таблиця показує конкретні об’єднання та вилучення. Сам факт зміни не доводить її правильності; дефісні назви та дві різні локації на кшталт `Київ-Одеса` потрібно розрізняти.

**Скільки рядків закінчуються службовим підписом TCH?**

Позначаємо рядки, що закінчуються `TCH` або `TCH.`. Рахуємо всі LOC-спани в цих рядках, не оголошуючи їх автоматично хибними.

In [12]:
uk["tch_footer"] = uk.text.str.contains(r"TCH\.?\s*$")
pd.Series({
    "рядків із TCH": int(uk.tch_footer.sum()),
    "усіх LOC-спанів у цих рядках": int(uk.loc[uk.tch_footer, "loc_markers"].map(len).sum()),
}).to_frame("кількість")

,кількість
рядків із TCH,9249
усіх LOC-спанів у цих рядках,9714


**Висновок.** Під шаблон потрапляють 9 249 рядків із 9 714 LOC-спанами. Ці числа описують рядки з підписом, а не кількість доведених помилок.

**Що саме містять рядки з TCH?**

Показуємо три повні тексти та їхні оригінальні LOC-мітки, щоб відрізнити зміст новини від службового хвоста.

In [13]:
footer_examples = uk.loc[uk.tch_footer, ["text", "loc_markers"]].head(3).copy()
footer_examples["locations"] = [[t[a:b] for a, b in spans] for t, spans in zip(footer_examples.text, footer_examples.loc_markers)]
with pd.option_context("display.max_colwidth", None):
    display(footer_examples[["text", "locations"]])

,text,locations
235,"Під час ""кривавого"" затемнення на Місяць впав метеорит - Наука та IT - TCH.",[Місяць]
405,Нічна різанина в центрі Одеси обірвала двоє життів - новини України - Укрaїнa - TCH.,"[Одеси, України, Укрaїнa]"
500,У супермаркеті Києва невідомий погрожував українськомовному покупцю - Київ - TCH.,"[Києва, Київ]"


**Висновок.** Разом зі службовими рубриками в таких рядках є корисні згадки з основного тексту. Видалення всього рядка прибере й їх; обрізання тільки футера потребує окремого перевіреного правила.

**Додатково: які теги має локальний похідний parquet?**

Це окрема перевірка локального `uk_geo_dataset_processed_v1.parquet`, а не твердження про CSV. Походження файла не встановлено. Якщо його немає, перевірка пропускається; решта розділу використовує CSV.

In [14]:
parquet_path = DATA / "uk_geo_dataset_processed_v1.parquet"
if parquet_path.exists():
    parquet_labels = pd.read_parquet(parquet_path, columns=["labels"])["labels"].head(5000)
    parquet_tags = Counter(tag for values in parquet_labels for tag in (json.loads(values) if isinstance(values, str) else values))
    display(pd.Series(parquet_tags).to_frame("токенів у перших 5 000 рядках"))
else:
    print("Локального parquet немає; перевірку пропущено.")

,токенів у перших 5 000 рядках
O,86060
LOC,2156


**Висновок.** У поточному локальному файлі перевірені теги — лише `O/LOC`. Вони не відрізняють дві сусідні локації від однієї багатослівної. У CSV межі є в `loc_markers`: з них будуємо `B-LOC` для початку кожної локації та `I-LOC` для її продовження.

### 1.1. Дублікати та перетини вибірок

Дублікат — повторення цілого тексту, а не назви міста в різних реченнях. Якщо текст трапляється тричі, це **одна група, три рядки в групі та дві зайві копії**.

Точний збіг порівнює оригінальні тексти. Нормалізований збіг ігнорує регістр і частину символів; він включає точні повтори, але не доводить взаємозамінність прикладів. Усередині train повтори збільшують вагу окремих текстів; між train і validation вони можуть завищувати оцінку.

**Що означає збіг після нормалізації?**

Переводимо текст у нижній регістр, замінюємо послідовності символів поза літерами, цифрами та `_` одним пробілом і обрізаємо крайові пробіли. Це ключ порівняння; оригінальний текст не змінюємо.

In [15]:
normalize_text = lambda s: re.sub(r"\W+", " ", s.lower()).strip()
uk["norm"] = uk.text.map(normalize_text)
normalization_examples = pd.Series(['«КИЇВ», сьогодні!', 'Київ — сьогодні', 'Івано-Франківськ', 'Київ', 'Києві'], name="оригінал")
pd.DataFrame({"оригінал": normalization_examples, "ключ": normalization_examples.map(normalize_text)})

,оригінал,ключ
0,"«КИЇВ», сьогодні!",київ сьогодні
1,Київ — сьогодні,київ сьогодні
2,Івано-Франківськ,івано франківськ
3,Київ,київ
4,Києві,києві


**Висновок.** Перші два приклади мають однаковий ключ. `Київ` і `Києві` лишаються різними: лематизації, перестановки слів і пошуку перефразувань тут немає.

**Скільки точних і нормалізованих повторів в uk?**

Рахуємо групи, всі рядки в групах і зайві копії окремо для всього uk, train та validation. `extra copies (%)` — частка зайвих копій від кількості рядків відповідної вибірки.

In [16]:
from IPython.display import display


def duplicate_summary(keys):
    sizes = keys.value_counts(dropna=False)
    repeated = sizes[sizes > 1]
    return {
        "rows": len(keys), "unique keys": len(sizes), "duplicate groups": len(repeated),
        "rows in duplicate groups": int(repeated.sum()),
        "extra copies": int((repeated - 1).sum()),
        "extra copies (%)": round(100 * (repeated - 1).sum() / max(len(keys), 1), 3),
        "largest group": int(sizes.max()) if len(sizes) else 0,
    }


assert duplicate_summary(pd.Series(["a", "a", "a", "b"]))["extra copies"] == 2
uk_duplicate_summary = pd.DataFrame({
    f"{split} / {match}": duplicate_summary(frame[key])
    for split, frame in {"uk all": uk, "uk train": uk[uk.is_valid.eq(0)], "uk validation": uk[uk.is_valid.eq(1)]}.items()
    for match, key in {"exact": "text", "normalised": "norm"}.items()
}).T
count_columns = uk_duplicate_summary.columns.drop("extra copies (%)")
uk_duplicate_summary[count_columns] = uk_duplicate_summary[count_columns].astype(int)
display(uk_duplicate_summary)

,rows,unique keys,duplicate groups,rows in duplicate groups,extra copies,extra copies (%),largest group
uk all / exact,1010000,1007546,2445,4899,2454,0.243,3
uk all / normalised,1010000,1006214,3749,7535,3786,0.375,4
uk train / exact,1000000,997598,2393,4795,2402,0.240,3
uk train / normalised,1000000,996294,3669,7375,3706,0.371,4
uk validation / exact,10000,10000,0,0,0,0.000,1
uk validation / normalised,10000,9999,1,2,1,0.010,2


**Висновок.** В uk — 2 454 зайві точні копії (0,243%) або 3 786 після нормалізації (0,375%). Друге число вже включає перше; додавати їх не потрібно.

**Скільки разів повторюється один текст?**

Для кожного розміру групи рахуємо кількість груп. Наприклад, рядок `copies in a group = 3` описує тексти, що трапляються тричі.

In [17]:
display(pd.DataFrame({
    name: uk[key].value_counts().loc[lambda s: s > 1].value_counts().sort_index()
    for name, key in {"exact": "text", "normalised": "norm"}.items()
}).fillna(0).astype(int).rename_axis("copies in a group"))

,exact,normalised
copies in a group,,
2,2436,3713
3,9,35
4,0,1


**Висновок.** Точні повтори мають максимум три копії; нормалізовані — чотири. Переважають пари: 2 436 точних груп і 3 713 нормалізованих.

**Як порівняти мітки, якщо пунктуація змістила координати?**

Відображаємо кожен LOC-спан у координати нормалізованого тексту. Зберігаємо окремі згадки та межі; спани без збережених символів рахуємо окремо. Короткі самоперевірки перевіряють зміщення, повторні згадки й об’єднання двох локацій.

In [18]:
def normalised_loc_signature(text, spans):
    positions, cursor = {}, 0
    for word in re.finditer(r"\w+", text.lower()):
        for i in range(word.start(), word.end()):
            positions[i] = (cursor, cursor + 1)
            cursor += 1
        cursor += 1  # One separator in the normalised text.
    signature, empty_spans = [], 0
    for a, b in spans:
        assert 0 <= a < b <= len(text)
        left, right = len(text[:a].lower()), len(text[:b].lower())
        retained = [positions[i] for i in range(left, right) if i in positions]
        if retained:
            signature.append((retained[0][0], retained[-1][1]))
        else:
            empty_spans += 1
    return tuple(sorted(signature)), empty_spans


assert normalised_loc_signature('«Київ»', [(1, 5)]) == normalised_loc_signature('КИЇВ', [(0, 4)])
assert normalised_loc_signature('Київ Київ', [(0, 4)]) != normalised_loc_signature('Київ Київ', [(5, 9)])
assert normalised_loc_signature('Київ Львів', [(0, 4), (5, 10)]) != normalised_loc_signature('Київ Львів', [(0, 10)])
assert normalised_loc_signature('-', [(0, 1)])[1] == 1
assert normalised_loc_signature('İ Київ', [(2, 6)]) == normalised_loc_signature('i Київ', [(2, 6)])


print("Усі 5 самоперевірок порівняння міток пройдено.")

Усі 5 самоперевірок порівняння міток пройдено.


**Висновок.** Метод розрізняє першу й другу згадку однакового міста та дві окремі локації проти одного спану. Збіг після нормалізації не доводить правильність розмітки або однакові межі пунктуації.

**Чи однаково розмічені повтори uk?**

Для однакового тексту порівнюємо оригінальні LOC-координати, для варіантів — їхнє відображення після нормалізації. Використовуємо `loc_markers` до очищення, щоб не приховати конфлікти.

In [19]:
uk_repeats = uk.loc[uk.norm.duplicated(keep=False)].copy()
uk_repeats['row_id'] = uk_repeats.index
uk_repeats['exact_labels'] = uk_repeats.loc_markers.map(lambda spans: tuple(sorted(spans)))
uk_repeats['normalised_labels'] = [normalised_loc_signature(t, s) for t, s in zip(uk_repeats.text, uk_repeats.loc_markers)]
uk_repeats['empty_loc_spans'] = uk_repeats.normalised_labels.map(lambda s: s[1])
uk_repeats['has_loc'] = uk_repeats.loc_markers.map(bool)
uk_repeats['locations'] = [[t[a:b] for a, b in spans] for t, spans in zip(uk_repeats.text, uk_repeats.loc_markers)]
uk_duplicate_groups = uk_repeats.groupby('norm', sort=False).agg(
    copies=('text', 'size'), text_versions=('text', 'nunique'),
    label_versions=('normalised_labels', 'nunique'), rows_with_loc=('has_loc', 'sum'),
    train_rows=('is_valid', lambda s: s.eq(0).sum()), valid_rows=('is_valid', 'sum'),
    footer_rows=('tch_footer', 'sum'), empty_loc_spans=('empty_loc_spans', 'sum'),
)
uk_exact_groups = uk_repeats.groupby('text', sort=False).agg(copies=('text', 'size'), label_versions=('exact_labels', 'nunique'))
uk_exact_groups = uk_exact_groups[uk_exact_groups.copies.gt(1)]
display(pd.Series({
    'exact duplicate groups with conflicting LOC offsets': int(uk_exact_groups.label_versions.gt(1).sum()),
    'normalised groups with several original texts': int(uk_duplicate_groups.text_versions.gt(1).sum()),
    'normalised groups with conflicting LOC signatures': int(uk_duplicate_groups.label_versions.gt(1).sum()),
    'normalised groups with punctuation-only LOC spans': int(uk_duplicate_groups.empty_loc_spans.gt(0).sum()),
}).to_frame('groups'))


,groups
exact duplicate groups with conflicting LOC offsets,0
normalised groups with several original texts,1319
normalised groups with conflicting LOC signatures,44
normalised groups with punctuation-only LOC spans,6


**Висновок.** Серед точних дублікатів конфліктів LOC немає. Серед нормалізованих груп — 44 з різними мітками; 1 319 груп містять різні оригінальні тексти. Такі варіанти не слід автоматично вважати рівноцінними.

**Які тексти uk повторюються найчастіше?**

Показуємо вісім найбільших нормалізованих груп. `copies` — кількість копій, `text_versions` — різних оригінальних текстів, `label_versions` — різних нормалізованих розміток.

In [20]:
uk_duplicate_groups.sort_values('copies', ascending=False).head(8)

,copies,text_versions,label_versions,rows_with_loc,train_rows,valid_rows,footer_rows,empty_loc_spans
norm,,,,,,,,
в україні різдво починають відзначати з свят вечора 6 січня разом з появою першої зірки в небі,4,2,1,4,4,0,0,0
ще один твір про який даша захотіла розповісти книга стіва мартіна так саме того популярного американського коміка радість мого суспільства,3,3,1,0,3,0,0,0
як завжди спостерігатимуться періоди більш холодної погоди,3,3,1,0,3,0,0,0
про це повідомили у відділі комунікації поліції харківської області,3,2,2,2,3,0,0,0
головне щоб колір плавно переходив з одного відтінку в інший,3,2,1,0,3,0,0,0
більше про економічні новини та курс валют в нашому огляді,3,2,1,0,3,0,0,0
навіщо прокурори коли є такі безвідмовні адвокати,3,3,1,0,3,0,0,0
в 11 класі сергій пішов з команди отримавши другий дорослий розряд з футболу,3,1,1,0,3,0,0,0


**Висновок.** Найбільша група має чотири рядки. За колонками видно, чи це точні копії, чи варіанти тексту й розмітки.

**Як виглядають точні повтори?**

Показуємо дві відповідні групи: повний текст, вихідні координати, вирізані локації та належність до validation. `row_id` — індекс рядка CSV.

In [21]:
keys = uk_duplicate_groups.loc[uk_duplicate_groups.text_versions.eq(1) & uk_duplicate_groups.rows_with_loc.gt(0)].head(2).index
with pd.option_context('display.max_colwidth', None):
    display(uk_repeats[uk_repeats.norm.isin(keys)][['row_id', 'is_valid', 'text', 'loc_markers', 'locations', 'tch_footer']])

,row_id,is_valid,text,loc_markers,locations,tch_footer
1378,1378,0,"Після закінчення інституту переїхав до Маріуполя, де протягом десяти років працював у Донецькому обласному російському драматичному театрі.","[(39, 48)]",[Маріуполя],False
1514,1514,0,У Покровську Донецької області 20-річного студента.,"[(2, 30)]",[Покровську Донецької області],False
397038,397038,0,"Після закінчення інституту переїхав до Маріуполя, де протягом десяти років працював у Донецькому обласному російському драматичному театрі.","[(39, 48)]",[Маріуполя],False
725856,725856,0,У Покровську Донецької області 20-річного студента.,"[(2, 30)]",[Покровську Донецької області],False


**Висновок.** Тексти й координати в показаних парах однакові. Для таких груп можна залишити одну копію; це не правило для конфліктних варіантів.

**Які відмінності приховує нормалізація?**

Показуємо дві відповідні групи: повний текст, вихідні координати, вирізані локації та належність до validation. `row_id` — індекс рядка CSV.

In [22]:
keys = uk_duplicate_groups.loc[uk_duplicate_groups.text_versions.gt(1) & uk_duplicate_groups.rows_with_loc.gt(0) & uk_duplicate_groups.label_versions.eq(1)].head(2).index
with pd.option_context('display.max_colwidth', None):
    display(uk_repeats[uk_repeats.norm.isin(keys)][['row_id', 'is_valid', 'text', 'loc_markers', 'locations', 'tch_footer']])

,row_id,is_valid,text,loc_markers,locations,tch_footer
1087,1087,0,На Закарпатті відбувся перший після послаблення карантину ультрамарафон «Карпатія» - чемпіонат України з гірського бігу на довгі дистанції.,"[(3, 13), (95, 102)]","[Закарпатті, України]",False
2692,2692,0,"Після того як потяг прибув до Києва, кривдника затримали правоохоронці.","[(30, 35)]",[Києва],False
539635,539635,0,"Після того, як потяг прибув до Києва, кривдника затримали правоохоронці.","[(31, 36)]",[Києва],False
717628,717628,0,"На Закарпатті відбувся перший після послаблення карантину ультрамарафон ""Карпатія"" - чемпіонат України з гірського бігу на довгі дистанції.","[(3, 13), (95, 102)]","[Закарпатті, України]",False


**Висновок.** Різні оригінальні тексти можуть мати однакові ключі та узгоджені локації. Нормалізований ключ потрібний для групування; текст і його координати зберігаємо разом.

**Що означає конфлікт міток?**

Показуємо дві відповідні групи: повний текст, вихідні координати, вирізані локації та належність до validation. `row_id` — індекс рядка CSV.

In [23]:
keys = uk_duplicate_groups.loc[uk_duplicate_groups.label_versions.gt(1)].head(2).index
with pd.option_context('display.max_colwidth', None):
    display(uk_repeats[uk_repeats.norm.isin(keys)][['row_id', 'is_valid', 'text', 'loc_markers', 'locations', 'tch_footer']])

,row_id,is_valid,text,loc_markers,locations,tch_footer
21630,21630,0,"Людина, яка працювала в Шахтарі 12 років, наговорила стільки про Динамо, що - ну як?","[(24, 31)]",[Шахтарі],False
28416,28416,0,"Нині офіційна влада Венесуели перекрила морський кордон з кількома карибськими островами, бо звідти до країни планували доправити гуманітарну допомогу.","[(20, 29)]",[Венесуели],False
33727,33727,0,"Людина, яка працювала в «Шахтарі» 12 років, наговорила стільки про «Динамо», що — ну як?",[],[],False
211377,211377,0,Нині офіційна влада Венесуели перекрила морський кордон з кількома Карибськими островами - бо звідти до країни планували доправити гуманітарну допомогу.,"[(20, 29), (67, 88)]","[Венесуели, Карибськими островами]",False


**Висновок.** У прикладах відрізняється наявність або склад LOC-міток. Залишити перший рядок означало б довільно вибрати одну версію; потрібна перевірка навчальної розмітки або відкладення всієї конфліктної групи.

**Чи трапляються тексти validation у train?**

Порівнюємо `is_valid=1` з `is_valid=0` за точним і нормалізованим текстом. Рахуємо саме рядки validation, а не групи.

In [24]:
train_texts = set(uk.loc[uk.is_valid.eq(0), "text"])
train_norm = set(uk.loc[uk.is_valid.eq(0), "norm"])
uk["leak"] = uk.is_valid.eq(1) & uk.norm.isin(train_norm)
validation = uk[uk.is_valid.eq(1)]
pd.Series({
    "точний збіг із train": int(validation.text.isin(train_texts).sum()),
    "збіг після нормалізації": int(uk.leak.sum()),
}).to_frame("рядків validation")

,рядків validation
точний збіг із train,52
збіг після нормалізації,79


**Висновок.** 52 рядки validation точно збігаються з train; ще 27 — після нормалізації. Разом 79 рядків, або 0,79% validation, слід вилучити перед оцінюванням.

**Які саме приклади потрапили в обидві частини?**

Показуємо дві нормалізовані групи, у яких є і train, і validation. Значення `is_valid` дозволяє перевірити перетин безпосередньо.

In [25]:
keys = uk_duplicate_groups.loc[uk_duplicate_groups.train_rows.gt(0) & uk_duplicate_groups.valid_rows.gt(0)].head(2).index
with pd.option_context("display.max_colwidth", None):
    display(uk_repeats[uk_repeats.norm.isin(keys)][["row_id", "is_valid", "text", "locations"]])

,row_id,is_valid,text,locations
1866,1866,0,"На іншій - ховається за своїми колінцями, обтягнутими панчохами-сіткою.",[]
6104,6104,0,Якщо ви хочете змінити своє життя – починайте вже сьогодні!,[]
1002041,1002041,1,"На іншій - ховається за своїми колінцями, обтягнутими панчохами-сіткою.",[]
1007264,1007264,1,Якщо ви хочете змінити своє життя – починайте вже сьогодні!,[]


**Висновок.** Копії одного тексту є по обидва боки поділу. Це перевірка на вже баченому вході; під час нового поділу всю групу потрібно тримати в одному split.

**Чи є повні тексти test в uk?**

Перевіряємо точний і нормалізований збіг цілого тексту. Читаємо тільки `test.text`, без перенесення або створення міток.

In [26]:
pd.Series({
    "точний збіг test з uk": int(test.text.isin(set(uk.text)).sum()),
    "нормалізований збіг test з uk": int(test.text.map(normalize_text).isin(set(uk.norm)).sum()),
}).to_frame("рядків test")

,рядків test
точний збіг test з uk,0
нормалізований збіг test з uk,0


**Висновок.** Повних збігів не виявлено. Це не виключає відредагованих репостів: змінені або додані слова дадуть інший ключ.

**Чи повторюються повні тексти всередині test?**

Рахуємо зайві точні та нормалізовані копії всередині test. Це окрема перевірка від перетину test із навчальними даними.

In [27]:
pd.Series({
    "зайві точні копії": int(test.text.duplicated().sum()),
    "зайві нормалізовані копії": int(test.text.map(normalize_text).duplicated().sum()),
}).to_frame("рядків test")

,рядків test
зайві точні копії,0
зайві нормалізовані копії,0


**Висновок.** Точних і нормалізованих повних повторів немає. Схожі, але не однакові тексти перевіряються окремо в розділі 4; тут їх не видаляємо.

**Відбираємо повтори ru для детального огляду.** Порівнюємо оригінальні тексти всіх рядків ru й залишаємо всі входження текстів, які повторюються. У цій же підготовці зберігаємо точні збіги з текстами test для окремої перевірки нижче; мітки test не використовуємо.

In [28]:
ru_repeats = ru.loc[ru.text.duplicated(keep=False), ["text", "loc_markers", "doc_id", "sent_id"]].copy()
ru_test_hits = ru.loc[ru.text.isin(set(test.text)), ["text", "doc_id", "sent_id"]].copy()
pd.Series({"Переглянуто рядків ru": len(ru), "Рядків із повторюваним текстом": len(ru_repeats)}).to_frame("Кількість")


,Кількість
Переглянуто рядків ru,8028840
Рядків із повторюваним текстом,772886


**Висновок.** Із 8 028 840 рядків відібрано 772 886, чиї тексти повторюються. Це всі входження, включно з першою копією кожного тексту; наступним кроком порахуємо групи й зайві копії.

**Скільки в ru точних повторів?** Групуємо відібрані рядки за оригінальним текстом. Окремо рахуємо групи, всі рядки в цих групах і зайві копії понад першу.

In [29]:
ru_repeats["marker_key"] = ru_repeats["loc_markers"].map(lambda spans: tuple(sorted(spans)))
ru_groups = ru_repeats.groupby("text", sort=False).agg(
    copies=("text", "size"), documents=("doc_id", "nunique"), label_versions=("marker_key", "nunique")
)
assert ru_groups["copies"].gt(1).all()
assert int(ru_groups["copies"].sum()) == len(ru_repeats)
ru_summary = pd.DataFrame([{
    "Усього рядків": len(ru),
    "Груп повторів": len(ru_groups),
    "Рядків у групах": int(ru_groups["copies"].sum()),
    "Зайвих копій": int((ru_groups["copies"] - 1).sum()),
    "Зайвих копій, %": round(100 * (ru_groups["copies"] - 1).sum() / len(ru), 3),
    "Найбільша група": int(ru_groups["copies"].max()),
}])
ru_summary

,Усього рядків,Груп повторів,Рядків у групах,Зайвих копій,"Зайвих копій, %",Найбільша група
0,8028840,249385,772886,523501,6.52,3916


**Висновок.** Підтверджено 249 385 груп із 772 886 рядками. Зайвих копій — 523 501, або 6,520% корпусу. Ці числа не можна трактувати як кількість помилково розмічених прикладів.

**Які тексти повторюються найчастіше?** Показуємо вісім найбільших груп, щоб зрозуміти зміст повторів, а не лише їхню кількість.

In [30]:
ru_groups.sort_values("copies", ascending=False).head(8).reset_index()[["text", "copies"]]

,text,copies
0,Об этом сообщает РИА Новости.,3916
1,Об этом сообщает Reuters.,3206
2,Об этом сообщает Associated Press.,2353
3,"Об этом сообщает ""Интерфакс"".",1838
4,Об этом сообщает корреспондент «Ленты.ру».,1635
5,Об этом сообщает «Интерфакс».,1439
6,Об этом сообщает Agence France-Presse.,1436
7,Об этом сообщает ТАСС.,1406


**Висновок.** Серед найчастіших повторів — шаблонні посилання на джерела новин. Наприклад, `Об этом сообщает РИА Новости.` трапляється 3 916 разів. Такі повтори збільшують вагу окремих шаблонів у train.

**Чи однакові мітки в точних копій ru?** Для кожного однакового тексту порівнюємо повний відсортований список оригінальних LOC-координат. Більше однієї версії означає конфлікт розмітки.

In [31]:
ru_conflicts = ru_groups[ru_groups["label_versions"].gt(1)]
pd.Series({
    "Груп із різними LOC-мітками": len(ru_conflicts),
    "Рядків у конфліктних групах": int(ru_conflicts["copies"].sum()),
}).to_frame("Кількість")

,Кількість
Груп із різними LOC-мітками,4786
Рядків у конфліктних групах,22562


**Висновок.** 4 786 груп, що містять 22 562 рядки, мають різні LOC-мітки для однакового тексту. Просте «залишити перший рядок» довільно вибере одну з версій розмітки.

**Як виглядають конфлікти ru?** Для трьох найбільших конфліктних груп показуємо по одному рядку кожної версії розмітки. `locations` — фрагменти, вирізані за наявними координатами; нових міток не створюємо.

In [32]:
ru_conflict_texts = ru_conflicts.sort_values("copies", ascending=False).head(3).index
ru_conflict_examples = ru_repeats[ru_repeats["text"].isin(ru_conflict_texts)].drop_duplicates(["text", "marker_key"]).copy()
ru_conflict_examples["locations"] = [
    [t[a:b] for a, b in spans] for t, spans in zip(ru_conflict_examples["text"], ru_conflict_examples["marker_key"])
]
with pd.option_context("display.max_colwidth", None):
    display(ru_conflict_examples[["text", "locations", "loc_markers", "doc_id", "sent_id"]])

,text,locations,loc_markers,doc_id,sent_id
2632,Об этом сообщает Associated Press.,[],[],227,1
83817,Киев отказался признать результаты плебисцита.,[Киев],"[(0, 4)]",7112,9
1632774,Об этом сообщает официальный сайт турнира.,[],[],147205,1
1752486,Об этом сообщает Associated Press.,[этом сообщает],"[(3, 16)]",157011,1
1753765,Киев отказался признать результаты плебисцита.,[],[],157111,12
1767306,Об этом сообщает официальный сайт турнира.,[сайт],"[(29, 33)]",158199,1


**Висновок.** У реченні `Киев отказался признать результаты плебисцита.` одна версія позначає `Киев`, а інша не містить локацій. Для першого бейзлайну такі групи можна відкласти до перевірки, замість вибору мітки навмання.

**Чи захистить поділ за `doc_id` від повторів між фолдами?** Рахуємо групи однакових речень, що трапляються більш ніж в одному документі.

In [33]:
pd.Series({
    "Груп повторів у кількох документах": int(ru_groups["documents"].gt(1).sum()),
    "Груп повторів лише в одному документі": int(ru_groups["documents"].eq(1).sum()),
}).to_frame("Кількість груп")

,Кількість груп
Груп повторів у кількох документах,240268
Груп повторів лише в одному документі,9117


**Висновок.** 240 268 груп повторів охоплюють кілька документів. Поділ лише за `doc_id` не гарантує відсутності однакових речень у train і validation: після поділу потрібна окрема перевірка збігів тексту.

**Чи є точні тексти test у повному ru?** Використовуємо збіги, зібрані під час повного проходу CSV. Це перевірка рівності оригінальних текстів, без нормалізації та без роботи з мітками test.

In [34]:
pd.Series({"Унікальних текстів test із точним збігом у ru": ru_test_hits["text"].nunique()}).to_frame("Кількість")

,Кількість
Унікальних текстів test із точним збігом у ru,0


**Висновок.** Точних збігів не знайдено. Нормалізовані збіги та відредаговані репости між ru й test цією перевіркою не охоплено.

**Чи перетинаються причини вилучення рядка?**

Кожен рядок відносимо до однієї комбінації трьох ознак: витік validation, футер TCH і зайва нормалізована копія за початковим порядком. Один рядок може мати кілька причин.

In [35]:
uk_removal_flags = pd.DataFrame({
    'validation overlaps train': uk.leak,
    'TCH footer': uk.tch_footer,
    'extra normalised copy (original row order)': uk.norm.duplicated(),
})
print('Overlapping reasons (each row appears in exactly one combination):')
display(uk_removal_flags[uk_removal_flags.any(axis=1)].value_counts().rename('rows').to_frame())

Overlapping reasons (each row appears in exactly one combination):


rows
validation overlaps train TCH footer extra normalised copy (original row order)      
False                     True       False                                       9226
                          False      True                                        3685
True                      False      True                                          78
False                     True       True                                          22
True                      True       True                                           1

**Висновок.** Причини перетинаються: зокрема, один рядок одночасно є витоком, має TCH і є зайвою копією. Тому 79, 9 249 і 3 786 не можна просто скласти.

**Скільки рядків залишиться після послідовних фільтрів?**

Це розрахунок, без вилучення рядків із `uk`: спочатку прибираємо витік validation, потім TCH, потім повтори всередині split. Порівнюємо точну дедуплікацію з агресивнішим нормалізованим сценарієм. `valid_mask` для наступних розділів виключає лише витік і TCH.

In [36]:
uk_removal_preview = []
for policy, key in {'exact copies': 'text', 'normalised copies (scenario)': 'norm'}.items():
    keep = pd.Series(True, index=uk.index)
    for stage in ['validation overlap', 'TCH footer', 'extra copies within each split']:
        candidates = uk.loc[keep, ["text", "norm", "is_valid", "leak", "tch_footer"]]
        if stage == 'validation overlap':
            remove = candidates.index[candidates.leak]
        elif stage == 'TCH footer':
            remove = candidates.index[candidates.tch_footer]
        else:
            remove = candidates.index[candidates.duplicated(['is_valid', key])]
        keep.loc[remove] = False
        uk_removal_preview.append({
            'policy': policy, 'stage': stage,
            'removed train': int(uk.loc[remove, 'is_valid'].eq(0).sum()),
            'removed validation': int(uk.loc[remove, 'is_valid'].eq(1).sum()),
            'remaining train': int((keep & uk.is_valid.eq(0)).sum()),
            'remaining validation': int((keep & uk.is_valid.eq(1)).sum()),
        })
    assert not uk.loc[keep].duplicated(['is_valid', key]).any()
    assert not set(uk.loc[keep & uk.is_valid.eq(0), 'norm']) & set(uk.loc[keep & uk.is_valid.eq(1), 'norm'])
display(pd.DataFrame(uk_removal_preview))
del candidates, keep
valid_mask = uk.is_valid.eq(1) & ~uk.leak & ~uk.tch_footer

,policy,stage,removed train,removed validation,remaining train,remaining validation
0,exact copies,validation overlap,0,79,1000000,9921
1,exact copies,TCH footer,9162,86,990838,9835
2,exact copies,extra copies within each split,2380,0,988458,9835
3,normalised copies (scenario),validation overlap,0,79,1000000,9921
4,normalised copies (scenario),TCH footer,9162,86,990838,9835
5,normalised copies (scenario),extra copies within each split,3684,1,987154,9834


**Висновок.** Таблиця рахує лише нові вилучення на кожному кроці. Після витоку й TCH у validation лишається 9 835 рядків; нормалізований сценарій прибирає ще один. Це сценарії обсягу даних: конфліктні групи ще потребують окремого рішення.

**Що беремо з розділу 1 у підготовку даних**

- Зберігати сирі CSV та координати; BIO будувати з окремих згадок у `loc_markers` після перевіреного очищення.
- Прибрати 79 рядків, що повторюють train, з наявної validation. Під час нового поділу тримати нормалізовану групу uk в одному split.
- Залишати одну точну копію за узгоджених міток. Конфліктні групи перевіряти або відкладати повністю; не вибирати першу розмітку навмання.
- Для ru після поділу за `doc_id` додатково перевіряти збіги речень між фолдами. Можна прибирати з train тексти, що є у validation, або групувати пов’язані повтореннями документи разом.
- Видалення цілих рядків TCH втрачає також корисний зміст. Це окремий компроміс, а не наслідок дедуплікації.

Ці перевірки описують дані та розмітку. Вплив очищення на F1 тут не вимірювався; моделі в цьому розділі не навчаються.

## 2. The target

The target is the set of location strings in a text. Question: what counts as a location in the silver train and in the manual sample, and where do the two policies differ?

In [37]:
STREET = r"(?i)\bвул|\bпроспект|\bбульвар|\bпров\.|\bпровул|\bплощ|\bузвіз|\bнабережн|\bулиц|\bпереул|\bплощад"
DISTRICT = r"(?i)район|р-н|област|\bобл\b|щин[аіиу]"
POI = r"\b(?:ТЦ|ТРЦ|ЖК|метро|Метро)\b"


def entity_table(df):
    per_text = df["spans"].map(len)
    strings = pd.Series([t[a:b] for t, s in zip(df["text"], df["spans"]) for a, b in s], dtype=str)
    return {
        "texts": len(df),
        "no LOC, share of texts": round((per_text == 0).mean(), 3),
        "entities per text, mean": round(per_text.mean(), 2),
        "entities per text with LOC, max": int(per_text.max()),
        "one-word entities": round((strings.str.split().str.len() == 1).mean(), 3),
        "street/address": round(strings.str.contains(STREET).mean(), 3),
        "district/region": round(strings.str.contains(DISTRICT).mean(), 3),
        "POI (ТЦ, ЖК, метро)": round(strings.str.contains(POI).mean(), 3),
        "street/address entities, count": int(strings.str.contains(STREET).sum()),
        "Україн*/Украин* (noun)": round(strings.str.fullmatch(r"(?i)украин\w*|україн\w*").mean(), 3),
    }


pd.DataFrame({"uk": entity_table(uk), "ru": entity_table(ru), "labeling_sample": entity_table(sample)})

,uk,ru,labeling_sample
texts,1010000.000,8028840.000,26.000
"no LOC, share of texts",0.769,0.693,0.385
"entities per text, mean",0.340,0.460,1.230
"entities per text with LOC, max",21.000,55.000,10.000
one-word entities,0.845,0.881,0.719
street/address,0.016,0.009,0.156
district/region,0.068,0.030,0.125
"POI (ТЦ, ЖК, метро)",0.000,0.000,0.031
"street/address entities, count",5519.000,32060.000,5.000
Україн*/Украин* (noun),0.160,0.037,0.000


In [38]:
# Does the type word ("вул.", "місті") go inside the span? Decides what the tagger must learn at the boundary.
STREET_WORD = re.compile(r"(?i)^(вул|вулиц\w*|ул|улиц\w*|проспект\w*|просп|площ\w*)$")
SETTLEMENT_WORD = re.compile(r"(?i)^(м|місто|міста|місті|село|села|селі|смт|город\w*|поселк\w*)$")
PREV_WORD = re.compile(r"(\w+)\.?\s*$")
FIRST_WORD = re.compile(r"\w+")


def type_word_position(df):
    c = Counter()
    for text, spans in zip(df["text"], df["spans"]):
        for a, b in spans:
            first = FIRST_WORD.search(text[a:b])
            prev = PREV_WORD.search(text[max(0, a - 20):a])
            first, prev = first.group() if first else "", prev.group(1) if prev else ""
            for name, pattern in [("street", STREET_WORD), ("settlement", SETTLEMENT_WORD)]:
                c[f"{name}: inside span"] += bool(pattern.match(first))
                c[f"{name}: before span"] += bool(pattern.match(prev))
    return c


pd.DataFrame({"uk": type_word_position(uk), "ru": type_word_position(ru), "labeling_sample": type_word_position(sample)}).fillna(0).astype(int)

,uk,ru,labeling_sample
street: inside span,4655,2483,5
street: before span,655,20479,0
settlement: inside span,358,283,0
settlement: before span,8320,84955,0


In [ ]:
# How consistently are the most test-relevant strings labelled? Share of whole-word occurrences.
def label_rates(df, pattern, org=True):
    rx = re.compile(pattern)
    c = Counter()
    org_col = df["org_markers"] if org and "org_markers" in df else [[]] * len(df)
    for text, spans, orgs in zip(df["text"], df["spans"], org_col):
        for m in rx.finditer(text):
            a, b = m.span()
            c["occurrences"] += 1
            if any(s <= a and b <= e for s, e in spans):
                c["LOC"] += 1
            elif any(s <= a and b <= e for s, e in orgs):
                c["inside ORG"] += 1
            else:
                c["unlabelled"] += 1
    return {k: (round(v / c["occurrences"], 3) if k != "occurrences" else v) for k, v in c.items()}


UKRAINE = r"\bУкраїн(?:а|и|і|у|ою)\b"
KYIV = r"\b(?:Київ|Києва|Києві|Києву|Києвом)\b"
pd.DataFrame({
    "uk: Україна": label_rates(uk, UKRAINE),
    "sample: Україна": label_rates(sample, UKRAINE, org=False),
    "uk: Київ": label_rates(uk, KYIV),
    "sample: Київ": label_rates(sample, KYIV, org=False),
}).fillna(0)

,uk: Україна,sample: Україна,uk: Київ,sample: Київ
occurrences,75065.000,4.0,15467.000,15.0
inside ORG,0.171,0.0,0.065,0.0
LOC,0.738,0.0,0.925,0.4
unlabelled,0.091,1.0,0.011,0.6


In [39]:
# How often do test posts mention what train barely labels? Occurrences per 1000 texts.
PATTERNS = {"Київ*": r"Київ|Києв", "район": r"район", "вул.": r"вул\.", "вулиц": r"вулиц", "проспект": r"проспект", "метро": r"метро", "ЖК/ТЦ": r"\b(?:ЖК|ТЦ)\b", "Україн*": r"Україн"}
pd.DataFrame({
    name: {k: round(1000 * df["text"].str.contains(p).mean(), 1) for k, p in PATTERNS.items()}
    for name, df in {"uk": uk, "ru": ru, "test": test, "labeling_sample": sample}.items()
})

,uk,ru,test,labeling_sample
Київ*,20.1,0.0,383.6,461.5
район,11.3,11.8,134.2,115.4
вул.,1.2,0.0,39.8,76.9
вулиц,5.7,0.0,44.0,38.5
проспект,0.5,1.0,23.1,38.5
метро,2.0,13.4,14.7,76.9
ЖК/ТЦ,0.1,0.3,14.7,76.9
Україн*,78.1,0.0,155.1,115.4


**Findings**
- Train and sample have different class mixes. Streets and addresses are 1.6% of uk locations but 15.6% in the sample. Train is country-heavy and test is about Kyiv: `Київ*` appears in 38% of test posts and 2% of uk sentences.
- uk still holds 5,519 street spans, so raising their share with existing sentences is a cheap first step.
- The type word sits inside the span in uk and the sample (`вул.`: 4,655 inside, 655 before). In the full ru corpus it is the opposite (2,483 inside, 20,479 before), so raw ru spans teach the wrong boundary for streets.
- `Україна` is labelled in 74% of uk occurrences and in 0 of 4 in the sample. `Київ` is labelled in only 40% of sample mentions, because channel signatures and institution names are not marked. The sample is small, but this is the largest false-positive risk.
- The sample has errors (`вул. Здолбунівська, 1` instead of `17`, skipped stations), so it is not exhaustive labelling.

## 3. The metric

Question: what does the metric reward, where does it break, and what do typical mistakes cost?

In [ ]:
cases = ["Київ", "м. Київ", "м.Київ", "Київ,", "Київ.", "(Київ)", "«Київ»", "#Київ", "Київ🫶", "Івано-Франківськ", "Івано - Франківськ",
         "вул. Солом'янська, 28", "вул. Солом'янська", "р-н Оболонь", "с. Гора", "Kyiv", "KYIV", "Odesa", "ODESA", "Nature reserve", "В", "28"]
pd.DataFrame({"raw": cases, "hard_processing": [hard_processing(c) for c in cases]})

,raw,hard_processing
0,Київ,київ
1,м. Київ,київ
2,м.Київ,мкиїв
3,"Київ,","київ,"
4,Київ.,київ
5,(Київ),київ
6,«Київ»,«київ»
7,#Київ,#київ
8,Київ🫶,київ🫶
9,Івано-Франківськ,івано-франківськ


In [ ]:
# Proposed alternative: same set logic, correct normalisation, partial credit for boundary errors.
def fixed_norm(s):
    s = unicodedata.normalize("NFKC", s).replace("’", "'").replace("ʼ", "'").lower()
    s = re.sub(r"[^\w\s']|\d", " ", s)
    s = re.sub(r"\b\w\b", " ", s)
    return " ".join(s.split())


def overlap_f1(y_true, y_pred):
    credit = n_true = n_pred = 0
    for t, p in zip(y_true, y_pred):
        t, p = {fixed_norm(s) for s in t} - {""}, {fixed_norm(s) for s in p} - {""}
        n_true, n_pred = n_true + len(t), n_pred + len(p)
        free = set(t)
        for s in sorted(p, key=len, reverse=True):
            score = lambda g: len(set(s.split()) & set(g.split())) / len(set(s.split()) | set(g.split()))
            best = max(free, key=score, default=None)
            if best is not None and score(best) > 0:
                credit += score(best)
                free.remove(best)
    return round(2 * credit / (n_true + n_pred), 4) if n_true + n_pred else 1.0


y_true = sample["locations"].tolist()
perturbations = {
    "perfect": y_true,
    "nothing predicted": [[] for _ in y_true],
    "lowercase": [[s.lower() for s in t] for t in y_true],
    "each string twice": [t + t for t in y_true],
    "prefix 'м. '": [["м. " + s for s in t] for t in y_true],
    "first word only": [[s.split()[0] for s in t] for t in y_true],
    "trailing comma": [[s + "," for s in t] for t in y_true],
    "one extra FP per text": [t + ["Марс"] for t in y_true],
}
pd.DataFrame({
    "official": {k: comp_metric(y_true, v) for k, v in perturbations.items()},
    "alternative": {k: overlap_f1(y_true, v) for k, v in perturbations.items()},
})

,official,alternative
perfect,1.000000,1.0000
nothing predicted,0.000000,0.0000
lowercase,1.000000,1.0000
each string twice,1.000000,1.0000
prefix 'м. ',1.000000,1.0000
first word only,0.718750,0.8594
trailing comma,0.000000,1.0000
one extra FP per text,0.711111,0.7111


In [ ]:
# Leaderboard noise: resample texts with sample's entity counts under a model with 70% recall.
# Assumed model quality, not a measurement; public size 224 assumes the 47% split of the 2025 edition.
rng = np.random.default_rng(0)
gold_counts = np.array([len({hard_processing(s) for s in t}) for t in y_true])
top_share = gold_counts.max() / gold_counts.sum()


def simulate(n_texts, recall=0.7, fp_per_entity=0.3, fp_empty=0.05):
    k = rng.choice(gold_counts, n_texts)
    tp = rng.binomial(k, recall)
    fp = rng.poisson(fp_per_entity * (k > 0) + fp_empty)
    return 2 * tp.sum() / max(2 * tp.sum() + fp.sum() + (k - tp).sum(), 1)


print(f"sample: {len(y_true)} texts, {gold_counts.sum()} unique strings, largest text = {top_share:.0%} of them")
pd.DataFrame({n: pd.Series([simulate(n) for _ in range(4000)]).describe(percentiles=[0.025, 0.975])[["mean", "std", "2.5%", "97.5%"]] for n in [26, 224, 477]}).round(3)

sample: 26 texts, 32 unique strings, largest text = 31% of them


,26,224,477
mean,0.735,0.740,0.740
std,0.068,0.022,0.015
2.5%,0.588,0.694,0.709
97.5%,0.852,0.780,0.769


**Strengths.** The metric is simple, needs no character offsets, ignores case and repeats, and matches the real output of the task (a list of strings).

**Weaknesses and edge cases**
- The regex is not an f-string, so punctuation is barely removed. A trailing comma destroys a match, so boundaries must be cleaned: `Київ,` versus `Київ` scores 0.
- The same regex deletes some lowercase Latin letters: `Odesa` becomes `od`, while `ODESA` stays `odesa`. Latin names depend on case.
- There is no lemmatisation: `Київ` and `Києві` are different strings, so we must return the surface form.
- A boundary error costs a false positive and a false negative at once (first word only gives 0.72 on the sample).
- A string of digits or a single letter normalises to an empty string and is still counted. Such predictions must be dropped.
- It is micro-averaged: one list post provides 31% of all sample strings, and any false positive on a text without locations is fully penalised.

**Complementary metrics for experiments:** token-level F1 for B/I (shows whether the model learns), exact span F1 (seqeval), recall by entity type (city, street, district), the share of location-free texts with a false positive, recall on unseen entities.

**Alternative target metric: `overlap_f1`.** The same set-based F1, with correct normalisation (all punctuation, apostrophes, no damage to Latin letters) and partial credit by token Jaccard. A trailing comma no longer zeroes the score (0 versus 1.0), and a boundary error is penalised proportionally (0.72 versus 0.86). This better reflects usefulness, since `вул. Солом'янська` and `вул. Солом'янська, 28` are the same address.

**Leaderboard noise.** Assuming 70% recall and 224 public texts (the 47% split of the 2025 edition), the 95% interval of F1 is about 0.69 to 0.78. On the 26 sample texts it is 0.59 to 0.85, so differences of a few hundredths cannot be seen there.

## 4. Groups, time and validation

Question: which group or time factors exist, and what validation do they call for? Train-versus-test separability is measured in `adversarial_validation.ipynb`.

In [ ]:
# Group factors: there is no timestamp or channel column, so channels and reposts are estimated from the text.
def shingles(text, k=4):
    words = TOKEN.findall(text.lower())
    return {tuple(words[i:i + k]) for i in range(max(len(words) - k + 1, 1))}


FOOTER_RE = re.compile(r"(?i)підписат(ись|ися)|подписаться|надіслати новину|запропонувати новину|пряма трансляція|життя🫶оболонь|^@\w+$")
is_footer = lambda line: len(line) < 80 and bool(FOOTER_RE.search(line))
sets = test["text"].map(shingles).tolist()
near = [(i, j) for i in range(len(sets)) for j in range(i + 1, len(sets)) if len(sets[i] & sets[j]) / max(len(sets[i] | sets[j]), 1) >= 0.8]
footers = Counter(line.strip() for t in test["text"] for line in set(t.split("\n")) if is_footer(line.strip()))
sents_per_doc = pd.Series(ru_docs).value_counts()
print(pd.Series({
    "test: near-duplicate pairs (Jaccard >= 0.8)": len(near),
    "test: texts in those pairs": len({i for pair in near for i in pair}),
    "test: posts with a channel footer": int(test["text"].map(lambda t: any(is_footer(l.strip()) for l in t.split("\n"))).sum()),
    "test: text_id strictly increasing": bool(test["text_id"].is_monotonic_increasing),
    "ru: documents": len(sents_per_doc),
    "ru: sentences per doc, median": float(sents_per_doc.median()),
}).to_string())
footers.most_common(10)

test: near-duplicate pairs (Jaccard >= 0.8)         8
test: texts in those pairs                         13
test: posts with a channel footer                 142
test: text_id strictly increasing                True
ru: documents                                  739190
ru: sentences per doc, median                    10.0


[('👉👉Запропонувати новину👈👈', 24),
 ('Підписатись на Дарниця Livе', 24),
 ('Хуйовий Київ | Надіслати новину', 20),
 ('Київ — пряма трансляція 👈', 18),
 ('🔷\xa0Підписатися на Telegram\xa0| Instagram\xa0| Facebook\xa0| TikTok', 18),
 ('Життя🫶Оболонь', 17),
 ('Надіслати новину', 17),
 ('Підписатися\xa0\xa0\xa0 |\xa0\xa0\xa0 Запропонувати новину', 13),
 ('@lossolomas_kyiv', 7),
 ('🔷 Підписатися на Telegram | Instagram | Facebook | TikTok', 6)]

**Findings**
- There is no time information: no dates in the data, and `text_id` only increases. A time-based split is impossible.
- Group factors: `doc_id` in ru (739,190 documents, median 10 sentences), normalised-text duplicates in uk, and in test the channel signatures (142 posts with a footer) and 8 near-duplicate pairs.

**Proposed validation**
1. The same folds for every experiment, grouped by normalised text (uk) and `doc_id` (ru), with an additional duplicate-text overlap check for ru (section 1.1). Silver folds show whether the model learns.
2. `labeling_sample` is too small for reliable threshold tuning; prefer using it as a sanity check over several seeds. This is a methodological recommendation, not a competition rule.
3. Policy decisions (streets, `Україна`, signatures) need validation beyond the small sample; submissions can provide additional evidence. Do not manually label test.

Correlation between CV and the leaderboard is expected to be weak because of the domain gap and the different labelling policy. It should be higher for tokenisation and cleaning changes, and lower for anything that depends on the share of streets.

## 5. Sequences for the RNN

Question: how long are the inputs, what is lost in tokenisation, and which context signals actually carry information?

In [40]:
length = lambda df: df["text"].map(lambda t: len(TOKEN.findall(t)))
lengths = {"uk": length(uk.sample(200_000, random_state=0)), "ru": length(ru), "test": length(test), "labeling_sample": length(sample)}
table = pd.DataFrame({k: v.describe(percentiles=[0.5, 0.9, 0.99])[["50%", "90%", "99%", "max"]] for k, v in lengths.items()}).round(0).astype(int)
EMOJI = "[\U0001F000-\U0001FAFF☀-➿]"
shape = lambda df: {
    "multi-line texts": round(df["text"].str.contains("\n").mean(), 3),
    "texts with emoji": round(df["text"].str.contains(EMOJI).mean(), 3),
    "emoji glued to a word": round(df["text"].str.contains(rf"\w{EMOJI}|{EMOJI}\w").mean(), 3),
    "texts with nbsp": round(df["text"].str.contains("\xa0").mean(), 3),
    "share of apostrophes that are ʼ": round(df["text"].str.count("ʼ").sum() / max(df["text"].str.count("['’ʼ]").sum(), 1), 3),
}
uk_shape = uk.sample(200_000, random_state=0)
display(table)
display(pd.DataFrame({"uk": shape(uk_shape), "ru": shape(ru), "test": shape(test), "labeling_sample": shape(sample)}))
pd.Series({f"max_len {c}": round(float((lengths["test"] <= c).mean()), 3) for c in [128, 256, 512, 700]}, name="test posts fully covered").to_frame()

,uk,ru,test,labeling_sample
50%,16,18,46,36
90%,29,32,148,144
99%,41,50,361,184
max,73,251,599,187


,uk,ru,test,labeling_sample
multi-line texts,0.000,0.000,0.851,0.808
texts with emoji,0.000,0.000,0.616,0.692
emoji glued to a word,0.000,0.000,0.321,0.346
texts with nbsp,0.000,0.000,0.130,0.077
share of apostrophes that are ʼ,0.003,0.001,0.178,0.143


,test posts fully covered
max_len 128,0.862
max_len 256,0.973
max_len 512,0.998
max_len 700,1.000


In [ ]:
# Ceiling: gold spans -> tokens -> tags -> strings. Everything lost here is lost before any model runs.
def ceiling(df, span_col, merge_adjacent=False):
    gold, pred, bad, total = [], [], 0, 0
    for text, spans in zip(df["text"], df[span_col]):
        offsets = tokenize(text)
        tags, misaligned = spans_to_bio(offsets, spans)
        bad, total = bad + misaligned, total + len(spans)
        gold.append([text[a:b] for a, b in spans])
        pred.append(bio_to_strings(text, offsets, tags, merge_adjacent))
    return {"F1 ceiling": comp_metric(gold, pred), "misaligned spans": round(bad / max(total, 1), 4)}


uk_part = uk.sample(200_000, random_state=0)
pd.DataFrame({
    "raw markers, BIO": ceiling(uk_part, "loc_markers"),
    "raw markers, IO": ceiling(uk_part, "loc_markers", merge_adjacent=True),
    "cleaned markers, BIO": ceiling(uk_part, "spans"),
    "cleaned markers, IO": ceiling(uk_part, "spans", merge_adjacent=True),
    "labeling_sample, BIO": ceiling(sample, "spans"),
})

,"raw markers, BIO","raw markers, IO","cleaned markers, BIO","cleaned markers, IO","labeling_sample, BIO"
F1 ceiling,0.995863,0.965779,0.995813,0.978619,1.0000
misaligned spans,0.004900,0.004900,0.004300,0.004300,0.0312


In [ ]:
# Which token-level signals predict LOC? Case, left and right context, on cleaned uk markers.
stats = {"case": Counter(), "left": Counter(), "right": Counter()}
loc_tok, other_tok = 0, 0
for text, spans in zip(uk_part["text"], uk_part["spans"]):
    offsets = tokenize(text)
    words = [text[a:b] for a, b in offsets]
    tags, _ = spans_to_bio(offsets, spans)
    for i, w in enumerate(words):
        if not w[0].isalpha():
            continue
        is_loc = tags[i] != "O"
        loc_tok, other_tok = loc_tok + is_loc, other_tok + (not is_loc)
        case = "ALLCAPS" if w.isupper() and len(w) > 1 else "Capitalized, sentence start" if w[0].isupper() and i == 0 else "Capitalized, mid-sentence" if w[0].isupper() else "lowercase"
        stats["case"][(case, is_loc)] += 1
        if tags[i] == "B":
            stats["left"][(words[i - 1].lower() if i else "<s>", True)] += 1
        elif not is_loc:
            stats["left"][(words[i - 1].lower() if i else "<s>", False)] += 1
        if is_loc and (i + 1 == len(tags) or tags[i + 1] != "I"):
            stats["right"][(words[i + 1].lower() if i + 1 < len(words) else "</s>", True)] += 1
        elif not is_loc:
            stats["right"][(words[i + 1].lower() if i + 1 < len(words) else "</s>", False)] += 1

base_rate = loc_tok / (loc_tok + other_tok)


def cue_table(counts, min_count=300, top=8):
    keys = {k for k, _ in counts}
    rows = [(k, counts[(k, True)], counts[(k, True)] / (counts[(k, True)] + counts[(k, False)])) for k in keys if counts[(k, True)] >= min_count]
    return pd.DataFrame(sorted(rows, key=lambda r: -r[2])[:top], columns=["cue", "LOC count", "P(LOC)"]).assign(lift=lambda d: (d["P(LOC)"] / base_rate).round(1)).round(3)


print("base rate of LOC tokens:", round(base_rate, 4))
display(cue_table(stats["case"], min_count=1))
display(cue_table(stats["left"]))
cue_table(stats["right"])

base rate of LOC tokens: 0.0287


,cue,LOC count,P(LOC),lift
0,"Capitalized, mid-sentence",68812,0.282,9.8
1,ALLCAPS,3578,0.119,4.1
2,"Capitalized, sentence start",987,0.005,0.2
3,lowercase,8041,0.003,0.1


,cue,LOC count,P(LOC),lift
0,місті,369,0.591,20.6
1,новини,326,0.556,19.4
2,території,385,0.413,14.4
3,міста,300,0.377,13.1
4,президент,541,0.326,11.4
5,президента,373,0.260,9.0
6,в,9800,0.204,7.1
7,проти,346,0.200,6.9


,cue,LOC count,P(LOC),lift
0,),682,0.118,4.1
1,(,737,0.111,3.9
2,та,2607,0.104,3.6
3,-,2445,0.077,2.7
4,—,422,0.074,2.6
5,–,746,0.073,2.5
6,.,12073,0.066,2.3
7,:,505,0.063,2.2


**Findings**
- Lengths differ: median 16 tokens in uk and 46 in test, p99 41 versus 361. A `max_len` of 512 fully covers 99.8% of test posts, so truncation can be avoided.
- 85% of test posts are multi-line and 62% contain emoji. Train has almost neither. Emoji is glued to words in 32% of posts, so the tokenizer must separate them. The apostrophe `ʼ` (18% of apostrophes in test, 0.3% in uk) needs normalising too.
- BIO versus IO: the F1 ceiling is 0.996 versus 0.979 on cleaned markers, so IO should not be used.
- Case: a capitalised word in mid-sentence is a location 28% of the time, a lowercase word 0.3%, the first word of a sentence 0.5%.
- Left context is stronger (`місті` ×20.6, `в` ×7.1) than right context (brackets, `та`, dashes ×2.5 to 4). Both directions carry signal, so a bidirectional RNN is justified.

## 6. Ideas from papers and competitions: what the data says

Each cell takes an idea from the literature and checks whether it is relevant to this data. No models are trained here.

In [42]:
# Pretrained embeddings and char layer (Lample 2016, Chiu & Nichols 2016): needed when target words are out of vocabulary.
vocab_uk = Counter(w.lower() for t in uk_part["text"] for w in TOKEN.findall(t) if w[0].isalpha())
vocab_ru = Counter(w.lower() for t in ru["text"] for w in TOKEN.findall(t) if w[0].isalpha())
frequent = {w for w, n in vocab_uk.items() if n >= 3}
gold_words = [w.lower() for t in sample["locations"] for s in t for w in TOKEN.findall(s) if w[0].isalpha()]
test_words = [w for t in test["text"] for w in TOKEN.findall(t) if w[0].isalpha()]
names = [w for t in test["text"] for line in t.split("\n") for w in TOKEN.findall(line)[1:] if w[0].isupper()]
oov = lambda ws, v: round(float(np.mean([w.lower() not in v for w in ws])), 3)
pd.DataFrame({
    "uk vocabulary": {"all test words": oov(test_words, vocab_uk), "capitalised test words (candidates)": oov(names, vocab_uk), "gold words in sample": oov(gold_words, vocab_uk)},
    "uk, frequency >= 3": {"all test words": oov(test_words, frequent), "capitalised test words (candidates)": oov(names, frequent), "gold words in sample": oov(gold_words, frequent)},
    "uk + ru": {"all test words": oov(test_words, set(vocab_uk) | set(vocab_ru)), "capitalised test words (candidates)": oov(names, set(vocab_uk) | set(vocab_ru)), "gold words in sample": oov(gold_words, set(vocab_uk) | set(vocab_ru))},
})

,uk vocabulary,"uk, frequency >= 3",uk + ru
all test words,0.083,0.124,0.044
capitalised test words (candidates),0.157,0.225,0.096
gold words in sample,0.244,0.317,0.146


In [ ]:
# Char layer (Ma & Hovy 2016, Yang 2018): do word endings identify a place among capitalised words?
capitalised = Counter()
for text, spans in zip(uk_part["text"], uk_part["spans"]):
    offsets = tokenize(text)
    tags, _ = spans_to_bio(offsets, spans)
    for i, (a, b) in enumerate(offsets[1:], 1):
        w = text[a:b]
        if w[0].isupper() and w[1:].islower() and len(w) > 4:
            capitalised[(w[-3:].lower(), tags[i] != "O")] += 1
ending_base = sum(v for (k, loc), v in capitalised.items() if loc) / sum(capitalised.values())
endings = {k for k, _ in capitalised}
rows = [(k, capitalised[(k, True)] + capitalised[(k, False)], capitalised[(k, True)] / (capitalised[(k, True)] + capitalised[(k, False)])) for k in endings if capitalised[(k, True)] + capitalised[(k, False)] >= 400]
table = pd.DataFrame(rows, columns=["last 3 letters", "words", "P(LOC)"]).sort_values("P(LOC)", ascending=False).round(3)
print(f"base rate among capitalised words: {ending_base:.3f}")
forms = Counter(t[a:b] for t, s in zip(uk["text"], uk["spans"]) for a, b in s if re.match(r"Ки[їє]в|Києв", t[a:b]) and " " not in t[a:b])
display(pd.concat([table.head(10), table.tail(5)]))
forms.most_common(8)

base rate among capitalised words: 0.315


,last 3 letters,words,P(LOC)
68,асі,680,0.990
35,їні,3373,0.982
10,їну,789,0.980
75,єві,1392,0.979
63,ьщі,503,0.948
0,ині,1653,0.946
29,иму,758,0.946
73,вці,430,0.923
6,усі,412,0.891
84,нії,1220,0.872


[('Києві', 6544),
 ('Києва', 4062),
 ('Київ', 2630),
 ('Києвом', 548),
 ('Київщині', 344),
 ('Києву', 191),
 ('Київщини', 118),
 ('Київській', 55)]

In [ ]:
# Gazetteers (Chiu & Nichols; warning from Coleridge): how far does a context-free lookup of train locations go?
loc_strings = Counter(tuple(w.lower() for w in TOKEN.findall(t[a:b])) for t, s in zip(uk["text"], uk["spans"]) for a, b in s)
lexicon = {k for k, n in loc_strings.items() if n >= 5 and k}
max_words = max(map(len, lexicon))


def lookup(text, exclude_ukraine=False):
    offsets, out, i = tokenize(text), [], 0
    words = [text[a:b] for a, b in offsets]
    while i < len(words):
        for n in range(min(max_words, len(words) - i), 0, -1):
            key = tuple(w.lower() for w in words[i:i + n])
            if key in lexicon and words[i][:1].isupper():
                out.append(text[offsets[i][0]:offsets[i + n - 1][1]])
                i += n
                break
        else:
            i += 1
    return [s for s in out if not (exclude_ukraine and re.fullmatch(r"(?i)україн\w*|украин\w*", s))]


def score(y_true, y_pred):
    tp = fp = fn = 0
    for t, p in zip(y_true, y_pred):
        t, p = {hard_processing(s) for s in t}, {hard_processing(s) for s in p}
        tp, fp, fn = tp + len(t & p), fp + len(p - t), fn + len(t - p)
    return {"F1": comp_metric(y_true, y_pred), "TP": tp, "FP": fp, "FN": fn}


val_df = uk[valid_mask]
val_gold = [[t[a:b] for a, b in s] for t, s in zip(val_df["text"], val_df["spans"])]
kept_lines = lambda t: "\n".join(l for l in t.split("\n") if not is_footer(l.strip()))
y_true = sample["locations"].tolist()
print("lexicon entries:", len(lexicon))
pd.DataFrame({
    "uk is_valid": score(val_gold, [lookup(t) for t in val_df["text"]]),
    "sample": score(y_true, [lookup(t) for t in sample["text"]]),
    "sample, no footers": score(y_true, [lookup(kept_lines(t)) for t in sample["text"]]),
    "sample, no footers, no Україна": score(y_true, [lookup(kept_lines(t), True) for t in sample["text"]]),
}).T

lexicon entries: 4793


,F1,TP,FP,FN
uk is_valid,0.669240,2601.0,2037.0,534.0
sample,0.395062,16.0,33.0,16.0
"sample, no footers",0.426667,16.0,27.0,16.0
"sample, no footers, no Україна",0.450704,16.0,23.0,16.0


In [ ]:
# Same-string aggregation (PII 1st): how many gold strings have several occurrences, so one confident hit is enough?
repeats = [len(re.findall(rf"(?<!\w){re.escape(s)}(?!\w)", t)) for t, gold in zip(sample["text"], sample["locations"]) for s in set(gold)]
repeated_names = []
for t in test["text"]:
    caps = Counter(w for line in t.split("\n") for w in TOKEN.findall(line)[1:] if w[0].isupper() and w[1:].islower() and len(w) > 3)
    repeated_names += list(caps.values())
pd.Series({
    "sample: gold strings": len(repeats),
    "sample: gold strings occurring more than once": int(sum(r > 1 for r in repeats)),
    "test: distinct capitalised words in a post": len(repeated_names),
    "test: of these, repeated in the same post": int(sum(r > 1 for r in repeated_names)),
})

sample: gold strings                               32
sample: gold strings occurring more than once       2
test: distinct capitalised words in a post       2738
test: of these, repeated in the same post         211
dtype: int64

In [43]:
# Multi-word and adjacent entities (CRF / BIO constraints, Lample 2016): how much structure is there to model?
def structure(df):
    c = Counter()
    for text, spans in zip(df["text"], df["spans"]):
        spans = sorted(spans)
        for (a1, b1), (a2, b2) in zip(spans, spans[1:]):
            gap = text[b1:a2]
            c["adjacent pairs, gap is whitespace"] += gap.strip() == ""
            c["adjacent pairs, gap is , or и/і/та"] += gap.strip().lower() in {",", "і", "й", "та", "и"}
        c["entities"] += len(spans)
        c["multi-token entities"] += sum(len(TOKEN.findall(text[a:b])) > 1 for a, b in spans)
    return c


pd.DataFrame({"uk": structure(uk_part), "ru": structure(ru), "labeling_sample": structure(sample)}).fillna(0).astype(int)

,uk,ru,labeling_sample
entities,67799,3706582,32
multi-token entities,11676,576439,9
"adjacent pairs, gap is whitespace",808,44917,0
"adjacent pairs, gap is , or и/і/та",5130,301483,2


**What the data says about ideas from the literature**

| Idea | Source | What the data shows | Pointer |
|---|---|---|---|
| Pretrained fastText | Lample 2016, Chiu & Nichols 2016 | 24% of gold location words and 16% of capitalised test words are not in the uk vocabulary | try first |
| Char layer | Ma & Hovy 2016, Yang 2018 | endings `-їні`, `-єві`, `-ьщі` give P(LOC) 0.95 to 0.98, and `Київ` has more than 6 frequent forms | worth trying |
| Case feature, lowercased training copies | Chiu & Nichols, Mayhew 2019 | mid-sentence capitalisation is a strong signal, but 4 of 32 gold strings start with a lowercase letter | worth trying |
| Bidirectional RNN | Ma & Hovy 2016 | signal exists in both directions | worth trying |
| CRF or BIO constraints | Lample 2016 | multi-token entities: 17% in uk, 28% in the sample | test it |
| More streets in train | Dai & Adel 2020, Coleridge 1st | streets 1.6% versus 15.6%, and the ru boundary is wrong | worth trying, the main gap |
| Location dictionary as a rule | Chiu & Nichols, warning from Coleridge | lookup alone gives 0.67 on silver and 0.40 on the sample (16 TP, 33 FP) | only as an extra feature |
| Merging identical strings in a text | PII 1st | only 2 of 32 sample strings repeat, 8% of capitalised words in test | low priority |
| Removing channel signatures | PII 1st, post-processing | 142 test posts, removes 6 of 33 false positives on the sample | worth trying |
| Partial CRF for noisy labels | Mayhew 2019 | `Київ` is unlabelled in only 1.1% of cases and `Україна` in 9.1%, so the problem is a policy difference, not random noise | low priority |
| Pseudo-labelling | Toxic 1st, Quora 1st | test has only 477 posts and the evidence is weak | not a priority |

The sample has 32 mentions, so its numbers are indicative and not measurements.

## 7. What to take into experiments

**Data preparation (required).** Build BIO from cleaned `loc_markers`. The tokenizer should split hyphens and emoji and normalise apostrophes without changing offsets. Remove validation leakage; deduplicate after checking annotation conflicts, following section 1.1. Handle TCH footers separately. Cut the output string from the original text, trim edge punctuation, and drop strings that are empty after normalisation.

**Worth trying, by decreasing expected value**
1. Pretrained fastText uk + ru.
2. A larger share of streets, districts and POI in train: existing sentences first, then mention replacement from an external street list.
3. A char layer and a case feature, training on original and lowercased text.
4. Post-like inputs: stitching sentences into multi-line texts, and predicting line by line.
5. Removing channel signatures before prediction.
6. CRF, or forbidding the `O → I` transition.
7. The `Україна` decision: by submission first, because the sample has only 4 mentions.
8. An ensemble over seeds and embeddings at the end.

**Limits of the Russian data.** Only 6.9% of test posts are in Russian and the ru street boundary is the opposite of uk, so ru should be added in limited amounts and checked separately.

**Stated competition rule.** “Manual labeling of test set is strictly forbidden!” No additional architecture or external-data restrictions are inferred here. Test text analysis does not create or correct test labels.

**Reference.** In the 2025 edition the RNN baseline scored 0.562, the BERT baseline 0.648, and the best teams 0.65 to 0.66.